# Fine-Tuning a Large Language Model for Layperson Medical Question Answering
AI in Healthcare | MSAI Spr '25 | Chelsea Ramos

## Objectives
1. ***Explore and sample the MedQuAD dataset***
2. Generate ~3.5k medical multiple-choice questions (MCQs) using Phi-2
3. Fine-tune TinyLLaMA with the generated MCQs
4. Evaluate model accuracy on held-out MCQs
5. Compare against other models (baseline and zero-shot)

---

# 1. Explore and Sample MedQuAD

In [1]:
import pandas as pd

## Load & Clean Data

- Dataset: MedQuAD 
    - 47,457 layperson medical question-answer pairs

In [2]:
df = pd.read_parquet("hf://datasets/lavita/MedQuAD/data/train-00000-of-00001-e36383d177026d53.parquet")
print('Shape:', df.shape)
df.head()

Shape: (47441, 13)


,document_id,document_source,document_url,category,umls_cui,umls_semantic_types,umls_semantic_group,synonyms,question_id,question_focus,question_type,question,answer
0,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-1,keratoderma with woolly hair,information,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...
1,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-2,keratoderma with woolly hair,frequency,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...
2,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-3,keratoderma with woolly hair,genetic changes,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen..."
3,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-4,keratoderma with woolly hair,inheritance,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...
4,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-5,keratoderma with woolly hair,treatment,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...


### Drop Missing Answers

In [3]:
df.isna().sum()

document_id                5
document_source            0
document_url               0
category               15431
umls_cui               16024
umls_semantic_types    16066
umls_semantic_group    16024
synonyms               22772
question_id                0
question_focus            14
question_type              0
question                   0
answer                 31034
dtype: int64

In [4]:
df.dropna(subset=['answer'], inplace=True)
print('Shape:', df.shape)

Shape: (16407, 13)


### Drop Duplicate QAs

In [5]:
print('Duplicates:', df.duplicated(subset=['question', 'answer']).sum())
df.drop_duplicates(subset=['question', 'answer'], inplace=True)
print('New Shape:', df.shape)

Duplicates: 48
New Shape: (16359, 13)


### Drop Small question_type Groups

In [6]:
df['question_type'].value_counts()

question_type
information        4520
symptoms           2747
treatment          2440
inheritance        1446
frequency          1120
genetic changes    1087
causes              708
exams and tests     650
research            395
outlook             361
susceptibility      324
considerations      228
prevention          209
stages               77
complications        46
support groups        1
Name: count, dtype: int64

In [7]:
small_groups = ['stages', 'complications', 'support groups', 'prevention', 'considerations']
df = df.loc[~(df['question_type'].isin(small_groups))]
print('Shape:', df.shape)

Shape: (15798, 13)


## Sample and Save 3.5k QAs

In [8]:
def balanced_group_sample(df, group_cols, n_per_group):
    # Group by specified cols
    grouped = df.groupby(group_cols)
    # Sample n rows (or all, if size is less than n) per group
    balanced_sample = grouped.apply(
        lambda x: x.sample(n=min(n_per_group, len(x)), random_state=42),
        include_groups=False
    ).reset_index()
    return balanced_sample

sampled_df = balanced_group_sample(df, group_cols=["question_type"], n_per_group=324)

# Check shape and distribution
print('Shape:', sampled_df.shape)
sampled_df['question_type'].value_counts()


Shape: (3564, 14)


question_type
causes             324
exams and tests    324
frequency          324
genetic changes    324
information        324
inheritance        324
outlook            324
research           324
susceptibility     324
symptoms           324
treatment          324
Name: count, dtype: int64

In [9]:
# Save sampled data to CSV
sampled_df.to_csv('../data/medquad_sampled.csv', index=False)
# Save QA only to jsonl
sampled_df = sampled_df[['question', 'answer']]
sampled_df.to_json('../data/medquad_sampled.jsonl', orient='records', lines=True)
sampled_df

,question,answer
0,What causes Cowden syndrome ?,What causes Cowden syndrome? Most cases of Cow...
1,What causes Non-classic congenital adrenal hyp...,What causes non-classic congenital adrenal hyp...
2,What causes Nephrocalcinosis ?,What causes nephrocalcinosis? Nephrocalcinosis...
3,What causes Sjogren syndrome ?,What causes Sjogren syndrome? Sjogren syndrome...
4,What causes Urinary Tract Infection In Adults ?,Most UTIs are caused by bacteria that live in ...
...,...,...
3559,What are the treatments for Landau-Kleffner Sy...,Treatment for LKS usually consists of medicati...
3560,What are the treatments for Norum disease ?,How might Norum disease be treated? Symptomati...
3561,What are the treatments for Meckel syndrome ?,These resources address the diagnosis or manag...
3562,What are the treatments for horizontal gaze pa...,These resources address the diagnosis or manag...
